In [2]:
# ==========================================
# Notebook 06
# Semantic Item Similarity Engine
# ==========================================

import pandas as pd
import numpy as np

import faiss

from sklearn.metrics.pairwise import cosine_similarity

In [3]:
profiles_df = pd.read_csv("../data/item_profiles.csv")

profiles_df.head()

,item_id,title,category,tags,content_profile
0,1,The Early Days of Stripe,Startup,startup fintech founders entrepreneurship vent...,Title: The Early Days of Stripe Category: Star...
1,2,Building SpaceX,Startup,startup fintech founders entrepreneurship vent...,Title: Building SpaceX Category: Startup Tags:...
2,3,AI for Healthcare,Artificial Intelligence,ai machine-learning llm deep-learning transfor...,Title: AI for Healthcare Category: Artificial ...
3,4,The Psychology of Habits,Wellness,habits productivity self-improvement,Title: The Psychology of Habits Category: Well...
4,5,Cloud Computing Fundamentals,Technology,cloud computing distributed-systems scalability,Title: Cloud Computing Fundamentals Category: ...


In [4]:
item_embeddings = np.load("../data/item_embeddings.npy")

item_embeddings.shape

(10, 384)

In [5]:
index = faiss.read_index("../data/item_index.faiss")

index.ntotal

0

In [6]:
print("Total Items:", len(profiles_df))

print()

print("Embedding Shape:", item_embeddings.shape)

Total Items: 10

Embedding Shape: (10, 384)


In [7]:
similarity_matrix = cosine_similarity(item_embeddings)

In [8]:
similarity_matrix.shape

(10, 10)

In [9]:
similarity_df = pd.DataFrame(
    similarity_matrix, index=profiles_df["title"], columns=profiles_df["title"]
)

similarity_df

title,The Early Days of Stripe,Building SpaceX,AI for Healthcare,The Psychology of Habits,Cloud Computing Fundamentals,Modern Data Engineering,Machine Learning in Finance,Startup Fundraising Guide,Nutrition Science,Large Language Models
title,,,,,,,,,,
The Early Days of Stripe,1.000000,0.604912,0.095936,0.202552,0.282320,0.214413,0.304935,0.626557,0.234726,0.100241
Building SpaceX,0.604912,1.000000,0.171755,0.201175,0.256445,0.302961,0.355282,0.791674,0.336658,0.207650
AI for Healthcare,0.095936,0.171755,1.000000,0.150197,0.263195,0.299612,0.489623,0.185238,0.344144,0.689363
The Psychology of Habits,0.202552,0.201175,0.150197,1.000000,0.169887,0.168850,0.213401,0.294368,0.584227,0.141652
Cloud Computing Fundamentals,0.282320,0.256445,0.263195,0.169887,1.000000,0.405300,0.294467,0.275134,0.261074,0.253823
Modern Data Engineering,0.214413,0.302961,0.299612,0.168850,0.405300,1.000000,0.387730,0.277132,0.350227,0.356685
Machine Learning in Finance,0.304935,0.355282,0.489623,0.213401,0.294467,0.387730,1.000000,0.448334,0.375995,0.462430
Startup Fundraising Guide,0.626557,0.791674,0.185238,0.294368,0.275134,0.277132,0.448334,1.000000,0.482051,0.203932
Nutrition Science,0.234726,0.336658,0.344144,0.584227,0.261074,0.350227,0.375995,0.482051,1.000000,0.240134


In [10]:
similarity_df.iloc[:5, :5]

title,The Early Days of Stripe,Building SpaceX,AI for Healthcare,The Psychology of Habits,Cloud Computing Fundamentals
title,,,,,
The Early Days of Stripe,1.000000,0.604912,0.095936,0.202552,0.282320
Building SpaceX,0.604912,1.000000,0.171755,0.201175,0.256445
AI for Healthcare,0.095936,0.171755,1.000000,0.150197,0.263195
The Psychology of Habits,0.202552,0.201175,0.150197,1.000000,0.169887
Cloud Computing Fundamentals,0.282320,0.256445,0.263195,0.169887,1.000000


In [11]:
def get_similar_items(item_title, top_k=5):

    item_row = profiles_df[profiles_df["title"].str.lower() == item_title.lower()]

    if len(item_row) == 0:

        return pd.DataFrame()

    item_index = item_row.index[0]

    scores = similarity_matrix[item_index]

    ranked_idx = np.argsort(scores)[::-1]

    results = []

    for idx in ranked_idx[1 : top_k + 1]:

        results.append(
            {
                "item_id": profiles_df.iloc[idx]["item_id"],
                "title": profiles_df.iloc[idx]["title"],
                "category": profiles_df.iloc[idx]["category"],
                "similarity": scores[idx],
            }
        )

    return pd.DataFrame(results)

In [12]:
get_similar_items("The Early Days of Stripe")

,item_id,title,category,similarity
0,8,Startup Fundraising Guide,Startup,0.626557
1,2,Building SpaceX,Startup,0.604912
2,7,Machine Learning in Finance,Machine Learning,0.304935
3,5,Cloud Computing Fundamentals,Technology,0.282320
4,9,Nutrition Science,Health,0.234726


In [13]:
get_similar_items("Building SpaceX")

,item_id,title,category,similarity
0,8,Startup Fundraising Guide,Startup,0.791674
1,1,The Early Days of Stripe,Startup,0.604912
2,7,Machine Learning in Finance,Machine Learning,0.355282
3,9,Nutrition Science,Health,0.336658
4,6,Modern Data Engineering,Data Engineering,0.302961


In [14]:
get_similar_items("Large Language Models")

,item_id,title,category,similarity
0,3,AI for Healthcare,Artificial Intelligence,0.689363
1,7,Machine Learning in Finance,Machine Learning,0.462430
2,6,Modern Data Engineering,Data Engineering,0.356685
3,5,Cloud Computing Fundamentals,Technology,0.253823
4,9,Nutrition Science,Health,0.240134


In [15]:
profiles_df[["title", "category"]]

,title,category
0,The Early Days of Stripe,Startup
1,Building SpaceX,Startup
2,AI for Healthcare,Artificial Intelligence
3,The Psychology of Habits,Wellness
4,Cloud Computing Fundamentals,Technology
5,Modern Data Engineering,Data Engineering
6,Machine Learning in Finance,Machine Learning
7,Startup Fundraising Guide,Startup
8,Nutrition Science,Health
9,Large Language Models,Artificial Intelligence


In [16]:
for title in profiles_df["title"].head(5):

    print()

    print("=" * 50)

    print(title)

    print("=" * 50)

    print()

    print(get_similar_items(title, top_k=3))


The Early Days of Stripe

   item_id                        title          category  similarity
0        8    Startup Fundraising Guide           Startup    0.626557
1        2              Building SpaceX           Startup    0.604912
2        7  Machine Learning in Finance  Machine Learning    0.304935

Building SpaceX

   item_id                        title          category  similarity
0        8    Startup Fundraising Guide           Startup    0.791674
1        1     The Early Days of Stripe           Startup    0.604912
2        7  Machine Learning in Finance  Machine Learning    0.355282

AI for Healthcare

   item_id                        title                 category  similarity
0       10        Large Language Models  Artificial Intelligence    0.689363
1        7  Machine Learning in Finance         Machine Learning    0.489623
2        9            Nutrition Science                   Health    0.344144

The Psychology of Habits

   item_id                        title 

In [17]:
def faiss_similarity_search(item_title, top_k=5):

    item_row = profiles_df[profiles_df["title"].str.lower() == item_title.lower()]

    if len(item_row) == 0:

        return pd.DataFrame()

    item_index = item_row.index[0]

    query_embedding = item_embeddings[item_index].astype("float32")

    distances, indices = index.search(query_embedding.reshape(1, -1), top_k + 1)

    results = []

    for rank, idx in enumerate(indices[0][1:]):

        results.append(
            {
                "rank": rank + 1,
                "item_id": profiles_df.iloc[idx]["item_id"],
                "title": profiles_df.iloc[idx]["title"],
                "category": profiles_df.iloc[idx]["category"],
                "distance": float(distances[0][rank + 1]),
            }
        )

    return pd.DataFrame(results)

In [18]:
faiss_similarity_search("The Early Days of Stripe")

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [19]:
faiss_similarity_search("Large Language Models")

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [20]:
faiss_results = faiss_similarity_search("Large Language Models")

faiss_results

,rank,item_id,title,category,distance
0,1,10,Large Language Models,Artificial Intelligence,3.402823e+38
1,2,10,Large Language Models,Artificial Intelligence,3.402823e+38
2,3,10,Large Language Models,Artificial Intelligence,3.402823e+38
3,4,10,Large Language Models,Artificial Intelligence,3.402823e+38
4,5,10,Large Language Models,Artificial Intelligence,3.402823e+38


In [21]:
recommendations = []

In [22]:
for title in profiles_df["title"]:

    similar_items = get_similar_items(title, top_k=3)

    for _, row in similar_items.iterrows():

        recommendations.append(
            {
                "source_item": title,
                "recommended_item": row["title"],
                "similarity": row["similarity"],
            }
        )

In [23]:
recommendations_df = pd.DataFrame(recommendations)

recommendations_df.head()

,source_item,recommended_item,similarity
0,The Early Days of Stripe,Startup Fundraising Guide,0.626557
1,The Early Days of Stripe,Building SpaceX,0.604912
2,The Early Days of Stripe,Machine Learning in Finance,0.304935
3,Building SpaceX,Startup Fundraising Guide,0.791674
4,Building SpaceX,The Early Days of Stripe,0.604912


In [24]:
recommendations_df.sort_values(by="similarity", ascending=False).head(20)

,source_item,recommended_item,similarity
3,Building SpaceX,Startup Fundraising Guide,0.791674
21,Startup Fundraising Guide,Building SpaceX,0.791674
27,Large Language Models,AI for Healthcare,0.689363
6,AI for Healthcare,Large Language Models,0.689363
0,The Early Days of Stripe,Startup Fundraising Guide,0.626557
22,Startup Fundraising Guide,The Early Days of Stripe,0.626557
1,The Early Days of Stripe,Building SpaceX,0.604912
4,Building SpaceX,The Early Days of Stripe,0.604912
9,The Psychology of Habits,Nutrition Science,0.584227
24,Nutrition Science,The Psychology of Habits,0.584227


In [25]:
def get_similar_by_item_id(item_id, top_k=5):

    item_row = profiles_df[profiles_df["item_id"] == item_id]

    if len(item_row) == 0:

        return pd.DataFrame()

    title = item_row.iloc[0]["title"]

    return get_similar_items(title, top_k)

In [26]:
get_similar_by_item_id(1)

,item_id,title,category,similarity
0,8,Startup Fundraising Guide,Startup,0.626557
1,2,Building SpaceX,Startup,0.604912
2,7,Machine Learning in Finance,Machine Learning,0.304935
3,5,Cloud Computing Fundamentals,Technology,0.282320
4,9,Nutrition Science,Health,0.234726


In [27]:
recommendations_df.to_csv("../data/item_similarity_recommendations.csv", index=False)

In [28]:
similarity_df.to_csv("../data/item_similarity_matrix.csv")

In [29]:
saved_df = pd.read_csv("../data/item_similarity_recommendations.csv")

saved_df.head()

,source_item,recommended_item,similarity
0,The Early Days of Stripe,Startup Fundraising Guide,0.626557
1,The Early Days of Stripe,Building SpaceX,0.604912
2,The Early Days of Stripe,Machine Learning in Finance,0.304935
3,Building SpaceX,Startup Fundraising Guide,0.791674
4,Building SpaceX,The Early Days of Stripe,0.604912
